In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

base_dir = r"C:\Users\Abhishek Shandilya\Desktop\suraksha"

print("All imports successful.")

All imports successful.


In [3]:
kaggle_path = os.path.join(base_dir, "data", "raw", "creditcard.csv")

df_kaggle = pd.read_csv(kaggle_path)

print("Kaggle dataset loaded:", df_kaggle.shape)
print("Columns:", list(df_kaggle.columns))
print("Fraud cases:", df_kaggle["Class"].sum())
print("Normal cases:", (df_kaggle["Class"] == 0).sum())
print("Fraud percentage:", round(df_kaggle["Class"].mean() * 100, 4), "%")

Kaggle dataset loaded: (284807, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Fraud cases: 492
Normal cases: 284315
Fraud percentage: 0.1727 %


In [4]:
iso_forest = joblib.load(os.path.join(base_dir, "models", "isolation_forest.pkl"))
scaler = joblib.load(os.path.join(base_dir, "models", "scaler.pkl"))
FEATURE_COLUMNS = joblib.load(os.path.join(base_dir, "models", "feature_columns.pkl"))

print("Model loaded successfully.")
print("Expected features:", FEATURE_COLUMNS)

Model loaded successfully.
Expected features: ['amount', 'km_from_last_txn', 'minutes_from_last_txn', 'impossible_speed', 'hour', 'is_weekend', 'is_night', 'txn_count_1hr', 'txn_count_24hr', 'amount_sum_1hr', 'amount_sum_24hr', 'amount_vs_user_avg', 'is_new_device', 'device_count_7d', 'is_new_merchant_category']


In [5]:
df_mapped = pd.DataFrame()

df_mapped['amount'] = df_kaggle['Amount']
df_mapped['hour'] = (df_kaggle['Time'] % 86400 / 3600).astype(int)
df_mapped['is_night'] = ((df_mapped['hour'] >= 23) | (df_mapped['hour'] <= 4)).astype(int)
df_mapped['is_weekend'] = 0
df_mapped['km_from_last_txn'] = df_kaggle['V1'].abs() * 10
df_mapped['minutes_from_last_txn'] = df_kaggle['Time'].diff().fillna(0) / 60
df_mapped['impossible_speed'] = (df_mapped['km_from_last_txn'] / (df_mapped['minutes_from_last_txn'].replace(0, 1) / 60) > 900).astype(int)
df_mapped['txn_count_1hr'] = df_kaggle['V2'].abs().round().astype(int)
df_mapped['txn_count_24hr'] = df_kaggle['V3'].abs().round().astype(int)
df_mapped['amount_sum_1hr'] = df_kaggle['Amount'] * df_mapped['txn_count_1hr']
df_mapped['amount_sum_24hr'] = df_kaggle['Amount'] * df_mapped['txn_count_24hr']
df_mapped['amount_vs_user_avg'] = df_kaggle['V4'].abs()
df_mapped['is_new_device'] = (df_kaggle['V5'].abs() > 1.5).astype(int)
df_mapped['device_count_7d'] = df_kaggle['V6'].abs().round().astype(int).clip(1, 10)
df_mapped['is_new_merchant_category'] = (df_kaggle['V7'].abs() > 1.5).astype(int)

df_mapped = df_mapped[FEATURE_COLUMNS]
y_kaggle = df_kaggle['Class'].values

print("Feature mapping complete.")
print("Mapped shape:", df_mapped.shape)
print("Null values:", df_mapped.isnull().sum().sum())
print("Fraud labels:", y_kaggle.sum())

Feature mapping complete.
Mapped shape: (284807, 15)
Null values: 0
Fraud labels: 492


In [6]:
print("Scaling features...")
X_kaggle_scaled = scaler.transform(df_mapped)

print("Running anomaly detection...")
kaggle_scores_raw = iso_forest.decision_function(X_kaggle_scaled)
kaggle_scores = (kaggle_scores_raw - kaggle_scores_raw.min()) / (kaggle_scores_raw.max() - kaggle_scores_raw.min())
kaggle_scores = 1 - kaggle_scores

kaggle_predictions = (iso_forest.predict(X_kaggle_scaled) == -1).astype(int)

print("Scoring complete.")
print()
print("Score statistics for real fraud transactions:")
print(pd.Series(kaggle_scores[y_kaggle == 1]).describe().round(4).to_string())
print()
print("Score statistics for real normal transactions:")
print(pd.Series(kaggle_scores[y_kaggle == 0]).describe().round(4).to_string())

Scaling features...
Running anomaly detection...
Scoring complete.

Score statistics for real fraud transactions:
count    492.0000
mean       0.6236
std        0.2110
min        0.0333
25%        0.5008
50%        0.6802
75%        0.7959
max        0.9028

Score statistics for real normal transactions:
count    284315.0000
mean          0.2452
std           0.1592
min           0.0000
25%           0.1331
50%           0.1977
75%           0.3230
max           1.0000


In [7]:
roc_auc = roc_auc_score(y_kaggle, kaggle_scores)

print("ROC-AUC Score on Real Kaggle Data:", round(roc_auc, 4))
print()
print("Classification Report:")
print(classification_report(y_kaggle, kaggle_predictions, target_names=["Normal", "Fraud"]))

cm = confusion_matrix(y_kaggle, kaggle_predictions)
print("Confusion Matrix:")
print("True Negatives  (Normal correctly identified):", cm[0][0])
print("False Positives (Normal flagged as fraud)    :", cm[0][1])
print("False Negatives (Fraud missed)               :", cm[1][0])
print("True Positives  (Fraud correctly caught)     :", cm[1][1])

ROC-AUC Score on Real Kaggle Data: 0.9016

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      0.92      0.96    284315
       Fraud       0.02      0.73      0.03       492

    accuracy                           0.92    284807
   macro avg       0.51      0.82      0.50    284807
weighted avg       1.00      0.92      0.96    284807

Confusion Matrix:
True Negatives  (Normal correctly identified): 262227
False Positives (Normal flagged as fraud)    : 22088
False Negatives (Fraud missed)               : 135
True Positives  (Fraud correctly caught)     : 357


In [8]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

scaler_kaggle = StandardScaler()
X_kaggle_scaled_new = scaler_kaggle.fit_transform(df_mapped)

iso_forest_kaggle = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination=0.0017,
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

print("Training on real Kaggle data...")
iso_forest_kaggle.fit(X_kaggle_scaled_new)
print("Training complete.")

kaggle_scores_new_raw = iso_forest_kaggle.decision_function(X_kaggle_scaled_new)
kaggle_scores_new = (kaggle_scores_new_raw - kaggle_scores_new_raw.min()) / (kaggle_scores_new_raw.max() - kaggle_scores_new_raw.min())
kaggle_scores_new = 1 - kaggle_scores_new

kaggle_predictions_new = (iso_forest_kaggle.predict(X_kaggle_scaled_new) == -1).astype(int)

roc_auc_new = roc_auc_score(y_kaggle, kaggle_scores_new)

print()
print("ROC-AUC Score (retrained on Kaggle):", round(roc_auc_new, 4))
print()
print("Classification Report:")
print(classification_report(y_kaggle, kaggle_predictions_new, target_names=["Normal", "Fraud"]))

cm_new = confusion_matrix(y_kaggle, kaggle_predictions_new)
print("Confusion Matrix:")
print("True Negatives  (Normal correctly identified):", cm_new[0][0])
print("False Positives (Normal flagged as fraud)    :", cm_new[0][1])
print("False Negatives (Fraud missed)               :", cm_new[1][0])
print("True Positives  (Fraud correctly caught)     :", cm_new[1][1])

Training on real Kaggle data...
Training complete.

ROC-AUC Score (retrained on Kaggle): 0.9004

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    284315
       Fraud       0.07      0.07      0.07       492

    accuracy                           1.00    284807
   macro avg       0.54      0.53      0.54    284807
weighted avg       1.00      1.00      1.00    284807

Confusion Matrix:
True Negatives  (Normal correctly identified): 283865
False Positives (Normal flagged as fraud)    : 450
False Negatives (Fraud missed)               : 457
True Positives  (Fraud correctly caught)     : 35


In [9]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

feature_cols_kaggle = ['Time', 'Amount', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6',
                       'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14',
                       'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22',
                       'V23', 'V24', 'V25', 'V26', 'V27', 'V28']

X_raw = df_kaggle[feature_cols_kaggle].values
y_raw = df_kaggle['Class'].values

scaler_raw = StandardScaler()
X_raw_scaled = scaler_raw.fit_transform(X_raw)

iso_raw = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination=0.0017,
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

print("Training directly on raw Kaggle features...")
iso_raw.fit(X_raw_scaled)
print("Training complete.")

scores_raw = iso_raw.decision_function(X_raw_scaled)
scores_normalized = (scores_raw - scores_raw.min()) / (scores_raw.max() - scores_raw.min())
scores_normalized = 1 - scores_normalized

predictions_raw = (iso_raw.predict(X_raw_scaled) == -1).astype(int)

roc_auc_raw = roc_auc_score(y_raw, scores_normalized)

print()
print("ROC-AUC Score (direct Kaggle features):", round(roc_auc_raw, 4))
print()
print("Classification Report:")
print(classification_report(y_raw, predictions_raw, target_names=["Normal", "Fraud"]))

cm_raw = confusion_matrix(y_raw, predictions_raw)
print("Confusion Matrix:")
print("True Negatives  (Normal correctly identified):", cm_raw[0][0])
print("False Positives (Normal flagged as fraud)    :", cm_raw[0][1])
print("False Negatives (Fraud missed)               :", cm_raw[1][0])
print("True Positives  (Fraud correctly caught)     :", cm_raw[1][1])

Training directly on raw Kaggle features...
Training complete.

ROC-AUC Score (direct Kaggle features): 0.9473

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    284315
       Fraud       0.26      0.26      0.26       492

    accuracy                           1.00    284807
   macro avg       0.63      0.63      0.63    284807
weighted avg       1.00      1.00      1.00    284807

Confusion Matrix:
True Negatives  (Normal correctly identified): 283958
False Positives (Normal flagged as fraud)    : 357
False Negatives (Fraud missed)               : 364
True Positives  (Fraud correctly caught)     : 128
